# RL general

Basado en RL: An Overview - Kevin P. Murphy

## Intro

Metodos para resolver **tareas de toma de decisiones secuenciales**.
- Agente que interactua con el entorno
- Agente mantiene un estado interno z_t
- Agente pasa el estado interno a la politica pi para seleccionar una accion: a_t = pi(z_t)
- El entorno devuelve una observacion o_t+1
- Agente actualiza su estado interno, usando una funcion de state-update: z_t+1 = SU(z_t, a_t, o_t+1)
    - Si asumimos que la observacion == verdadero world state w_t, podemos setear el estado interno del agente y el estado del entorno con la misma letra, s_t


**Maximum expected utility** principle

El objetivo del agente es **seleccionar la politica pi que maximice la suma de rewards esperados**:

**Value function**

$$V^{\pi}(s_0) = \mathbb{E}_{p(a_0, s_1, a_1, \ldots, a_T, s_T \mid s_0, \pi)} \left[ \sum_{t=0}^{T} R(s_t, a_t) \,\Big|\, s_0 \right]$$

- s_0 es el estado inicial del agente
- R(s_t, a_t) es el REWARD FUNCTION que el agente usa para medir el valor de realizar una accion en un estado
- V_pi(s_0) es el value function para la politica pi, evaluado en s_0

Expectation: 
- Cuando ponemos E y como subindice una distribucion, estamos diciendo como vamos a ponderar cada ejemplo
- Lo que esta adentro de corchetes (la suma de rewards) es una trayectoria particular (vamos sampleando de distribuciones segun los inputs y armamos una trayectoria) y eso lo sumamos y queda un reward por trayectoria (que es lo que vamos a promediar ponderadamente) -> rollout
- Ponderacion: p(a_0, s_1, a_1,..., a_T, s_T | s_0, pi). Es la probabilidad conjunta (de todas las combinaciones -> trayectorias t) de a_0, s_1, a_1,..., a_T, s_T dados la politica y el estado inicial.

$$p(\tau \mid s_0, \pi) = \prod_{t=0}^{T-1} \left[ \pi(a_t \mid s_t) \; p_{\text{env}}(o_{t+1} \mid \cdot) \; \delta(s_{t+1} = U(s_t, a_t, o_{t+1})) \right]$$

- **s es el estado interno** del agente, mientras que o son las observaciones del mundo
- Cada trayectoria t (a0, s1, a1,..., aT, sT) tiene un prob de ocurrir segun la distribucion conjunta (dado pi y s_0). Cada linea tiene 3 factores: politica x entorno x estado (actualiza s deterministicamente con delta de dirac)
    - pi(a_p|s_0): prob de seleccionar acciones dado el estado inicial
    - p_env(o_1|a_0): prob de ver observaciones dada la accion. Usualmente desconocido
    - delta(s = U()) (delta de dirac): es deterministica. Es como un one-hot conceptualmente, donde solo deja los valores que coinciden con U (los que son posibles), para los demas queda valor 0 y mata toda esa posiblidad (en el caso discreto).
- En la practica no calculamos toda la distribucion de prob conjunta y ponderamos por eso. Usamos monte carlo, vamos sampleando las trayectorias segun las distrib de prob particulares y despues dividimos por N simplemente -> si es on-policy!

**Politica optima**

$$\pi^* = \underset{\pi}{\arg\max} \; \mathbb{E}_{p_0(s_0)} \left[ V^{\pi}(s_0) \right]$$

Seleccionar el pi que maximice el valor esperado (sobre los estados iniciales segun la distribucion inicial del entorno).
- Cuando empezamos un episodio de RL, no siempre empezamos desde el mismo estado
- El entorno define una distribucion inicial de estados (esto es teorico matematico, asumimos que el entorno te devuelve un estado inicial al azar con ciertas probabilidades). No es que de verdad vamos a buscar en todos los estados iniciales posibles y ponderados segun una distribucion de probabilidad que nos da el entorno para elegir el estado inicial
- Igual podria ser, en el caso de un juego en el que empieces, por ejemplo, como local o visitante en un partido. Ahi el entorno (el juego) nos da una prob de distribucion sobre los estados iniciales (local 30% y visitante 70% por ejemplo)
$$\mathbb{E}_{p_0(s_0)}[V^{\pi}(s_0)] = 0.3\, V^{\pi}(\text{local}) + 0.7\, V^{\pi}(\text{visitante})$$
- Y la política óptima pi star sería la que maximiza ese promedio ponderado, que rinda bien en promedio según cómo te toque empezar el partido.
- Si siempre arrancaramos desde el mismo s_0, no buscariamos ninguna expectation aca y quedaria:
$$\pi^* = \underset{\pi}{\arg\max} \; V^{\pi}(s_0)$$


**Episodios**

- Continual -> no termina. Usamos el average reward porque no termina, entonces no sabemos como hacerlo mejor
- Episodic -> termina

Para el episodic, cada trayectoria es un episodio. Tiene un terminal state. La longitud del episodio es variable casi siempre. Si es fijo y conocido se llama finite horizon.

**RETURN G**

Para un estado en tiempo t, el estado es la suma de los rewards futuras desde el tiempo t (usando un discount factor).
$$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots + \gamma^{T-t-1} r_{T-1}$$

**VALUE FUNCTION** 

Valor ESPERADO del return futuro, segun la politica pi
$$V^{\pi}(s_t) = \mathbb{E}[G_t \mid \pi]$$

- Return es una muestra -> una trayectoria particular. Cuanto gane en esta partida, a partir de ese momento.
- Value function -> es el valor esperado de esos returns (segun la probabilidad de cada cosa). En promedio, cuanto esperaria ganar desde ese momento si juego muchas veces asi (sobre todas las posibles trayectorias)
    - Si tuvieramos la distribucion conjunta real de todo lo que puede pasar (estados, observaciones, acciones, etc), podemos calcular el reward esperado, mirando el return para cada trayectoria posible
    - Como es dificil tenerlo, podemos aproximarlo con monte carlo (sampleando muestras de trayectorias)

Discount Factor
- El discount factor sirve para que: el return sea finito incluso si T (tiempo fin) sea infinito. Y tambien que se le de mas importancia a lo mas pronto.
    - Si es muy chico, va a ser muy greedy y no pensar en largo plazo
    - Si es 0, solamente se va a enfocar en maximizar el reward inmediato
- Tambien se puede ver como que refleja que hay una prob de 1-gamma de que se termine el episodio en el prox paso. 
    - Si se que hay cierta chance de que mañana muera, voy a darle menos importancia al reward de dentro dos dias. Y mucho menos de dentro de 100 años
    - El valor esperado del tiempo de vida del agente (samplear muchas veces con cada paso con su gamma e ir viendo cuando muere) seria: E[T] = 1/(1-gamma)
    - Matemáticamente, el discounting y la probabilidad de terminar son equivalentes: Si en cada paso existe una chance 1−gamma de que el juego se corte, entonces el valor esperado de recompensas que obtenés es igual al del caso con descuento por gamma.


**Internal state z_t**

- A veces no es lo mismo lo observado con el modelo del mundo real, sino que el mundo tiene hidden states w_t (que se van actualizando segun a_t) con un transition function M(w_t,a_t,eps_t) (posiblemente desconocido para el agente).
- El agente no ve w_t pero si ve o_t+1, que es una observacion parcial o potencialmente ruidosa (eps_t+1)
- El agente usa las observaciones para crear un **internal belief state** (estado interno de creencia) sobre el mundo: z. Se actualiza con una funcion de actualizacion
$$z_{t+1} = SU(z_t, a_t, o_{t+1})$$

Esta es la informacion que se tiene del mundo (interna del agente). Pueden ser:
- No parametrico: por ejemplo guardar todas las observaciones pasadas
- Parametrico: aproximacion con un modelo (hidden state)

El estado interno del agente (state-update function) se puede dividir en dos:
- Prediction function P: predice lo que va a pasar dado el estado actual y la accion (imagina). World model. z_t+1 = P(z_t,a_t)
- Update function U: despues actualiza el estado interno, dado lo esperado segun P e incorporando lo real observado.

$$z_{t+1} = U(P(z_t, a_t), o_{t+1})$$

- Las observaciones se pueden codificar con un encoder E(o_t+1), por ejemplo para pasar de datos con muchas dimensiones (como imagenes) a un espacio latente.
- Tambien se puede entrenar un decoder para predecir la siguiente observacion: o^_t+1​ = D(z_t+1 ∣ t​) (esto no lo usamos para tomar acciones, sino para probar si nuestro world model funciona ok)

Teniendo un encoder y decoder, lo que podriamos hacer es: 
- Predecir el estado del mundo siguiente con P y usar D para convertirlo en una observacion predicha
- Comparar la observacion predicha con la observacion real (que la deberiamos tener)
- Que esto se convierta en una señal para entrenar el modelo.

**Aprendemos la politica de accion**: pi_t(z_t) = pi(z_t, **theta_t**), usando un algoritmo para **actualizar los parametros** de la politica:

$$\theta_t = A(o_{1:t}, a_{1:t}, r_{1:t}) = A(\theta_{t-1}, a_t, z_t, r_t)$$

- Empezando de cero tenemos: o1:t​,a1:t​,r1:t -> todas las observaciones, acciones y recompensas que el agente vio hasta el tiempo t
- En la practica no usamos toda la historia la vez, sino que usamos el modelo actualizado hasta el ultimo step y ahi tomamos la accion, estado y recompensa actual (version online). Usa su conocimiento anterior (con los theta hasta el momento), lo que el agente sabe del mundo (habria resumen de las observaciones), lo que hizo ahora y el resultado ahora (feedback, reward). Con eso vuelve a actualizar theta.

---

## RL approaches

Problema principal: **Encontrar la policy optima cuando el modelo del entorno es desconocido**. Queremos aprender a actuar sin conocer el entorno.

Dimensiones:
- Que aprende el agente? Value-based, policy-based, model-based (o alguna combinacion)
- Como el agente representa sus funciones desconocidas? non-parametric o parametric
- Como se seleccionan las acciones? On-policy (current agent's policy) o off-policy (cualquier politica externa)

---

### Value-based RL

(Approximate Dynamic Programming - ADP). En vez de aprender directamente que hacer, aprendemos **cuanto vale cada situacion o accion**. Despues, **elegimos la accion que con mas valor** para cada situacion.

**Value function**: si estoy en estado s y sigo la politica pi, cual es el promedio de todas las recompensas futuras? Ponderada por factor de descuento:

$$V_\pi(s) = \mathbb{E}_\pi \left[ \sum_{t=0}^{\infty} \gamma^t r_t \;\middle|\; s_0 = s \right]$$

**Bellman Equation**

Es la forma recursiva de verlo (y de programacion dinamica). Dice: El valor de un estado es igual al reward que consigue ahora mas el valor esperado del proximo estado, siguiendo siempre la mejor accion posible en siguientes pasos.

$$V^*(s) = \max_a \left[ R(s, a) + \gamma \mathbb{E}_{s' \sim p_S(\cdot|s,a)} \left[ V^*(s') \right] \right]$$

- R(s,a): es la recompensa inmediata de hacer la accion a en el estado s
- s': es el estado siguiente despues de tomar la accion a en el estado s. 
    - Pero NO ES DETERMINISTICO. Por eso usamos la expectation sobre ps(s'|s,a), que te dice cual es la prob de llegar a cada posible siguiente estado
- Expectation: como no sabemos cual es el siguiente estado, calculamos el promedio ponderado por la prob de que suceda cada uno (despues de tomar la accion a)

Si expandimos una vez bellman equation tenemos:

$$V^*(s) = \max_a \left[ R(s, a) + \gamma \mathbb{E}_{s'} \left[ \max_{a'} \left( R(s', a') + \gamma \mathbb{E}_{s''} [ V^*(s'') ] \right) \right] \right]$$

Factor de descuento: Es una cadena infinita de expectativas pero converge gracias al factor de descuento. Sigue de la misma manera, disminuyendo la importancia por paso, porque al expandir, si despues distribuis el factor, queda gamma al cuadrado y despues gamma al cubo, etc.

- El γ que estaba afuera multiplica todo lo que está adentro de la expectativa. Y ahi aparece el cuadrado:

$$V^*(s) = \max_a \left[ R(s, a) + \gamma \mathbb{E}_{s'} \left[ R(s', a') \right] + \gamma \cdot \gamma \mathbb{E}_{s'', \ldots} \left[ V^*(s'') \right] \right]$$

**Temporal Difference (TD Learning)**

Bellman equation es recontra teorico, porque para saber el valor real de un estado necesitariamos saber todo el entorno, como cada accion afecta al entorno y los rewards que tienen, como es el modelo de transicion, etc, para tomar las mejores decisiones y asi ver el pi optimo y sus rewards segun las transiciones... En la practica no lo sabemos. Entonces **aprendemos por experiencia** para estimar la Bellman Equation, usando muestras reales.

Paso a paso, vamos actualizando V(s) -> siempre para el mismo s eh... lo que queremos es actualizar el valor del estado s, usando estados futuros mientras vamos avanzando.
- Estaba en s, tengo mi V(s) actual estimado: yo creia que esto valia 10 puntos
- Hice una accion y obtuve un reward de 2 puntos y llegue a un estado s' que creo que vale 15, mi V(s')
- Ahora que se algo del futuro, puedo actualizar mi estimacion de V(s)

$$V(s) \leftarrow V(s) + \eta \left[ r + \gamma V(s') - V(s) \right]$$

Se llama Temporal Difference porqeu estoy aprendiendo por la diferencia entre dos predicciones en el tiempo. 

Siempre queremos estimar **cuanto vamos a ganar** en total desde s. Pero no lo sabemos, solo vemos:
- El reward inmediato r
- A que estado nos fuimos (y cuanto vale para nosotros, es mejor o peor?)
Entonces tenemos:
- **Prediccion vieja del valor de s**: V(s) -> lo que creia antes
- **Prediccion nueva del valor de s**: r + gamma * V(s') -> esta es nuestra nueva mejor estimacion de lo que valia s (actualizada con r y s')
- La diferencia es el **error temporal** (TD error): comparo la prediccion nueva con la vieja: $\left[ r + \gamma V(s') - V(s) \right]$
- Ese error (lo que yo creia de s, y la nueva creencia de s (teniendo en cuenta r y el valor que pienso que tiene s'))es lo que uso para corregir.
    - Si yo antes creia que s valia menos que lo que pienso ahora (en el futuro), entonces le aumento el valor a s. O sea, si r+gamma*V(s') es mayor a V(s), el valor de V(s) tiene que aumentar, porque lo estoy subestimando porque r y V(s') son mejores de lo que pensaba
    - Si r+gamma*V(s') es menor a V(s), entonces yo pensaba que V(s) era mejor de lo que es... entonces lo bajo un poco, porque o el reward visto no es tan bueno como esperaba o te lleva a un estado que no vale tanto en el futuro

Esto tiene sentido porque partimos de la value function, que nos dice que el valor de un estado es el reward de ese estado mas el valor futuro descontado... Esto es lo mismo que la prediccion nueva del valor de s en TD learning.
- En el caso ideal, se cumpliria que V(s) = r+gamma(V(s')) y el error TD seria 0. Lo que queremos es que se cumpla esa igualdad.

Si quiero usar mas pasos para actualizar V(s), uso multi-step TD:
- Mi nueva prediccion de V(s) seria:

$$\text{target de 2 pasos} = r_0 + \gamma r_1 + \gamma^2 V(s_2)$$

- Y mi actualizacion seria:
$$V(s_0) \leftarrow V(s_0) + \eta \left[ \left(r_0 + \gamma r_1 + \gamma^2 V(s_2)\right) - V(s_0) \right]$$

Y el n-step seria:

$$G_t^{(n)} = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots + \gamma^{n-1} r_{t+n-1} + \gamma^n V(s_{t+n})$$
$$V(s_t) \leftarrow V(s_t) + \eta \left[ G_t^{(n)} - V(s_t) \right]$$


**Q-learning**

Hasta ahora sabemos como estimar que tan bueno es estar en un estado s. Pero queremos tomar decisiones. **Si estoy en el estado s, que accion me conviene tomar?**.

Q-function (state-action value function):

$$Q^{\pi}(s, a) = \mathbb{E}_{\pi} \left[ \sum_{t=0}^{\infty} \gamma^t r_t \;\middle|\; s_0 = s, a_0 = a \right]$$

Cuanto espero ganar si tomo la accion a en el estado s? (Y si despues sigo con mi politica pi).
- Arrancamos con s_0 = s -> el estado actual
- Arrancamos con a -> una accion puntual que decidimos probar cuanto vale
- la suma es sobre t, que son los pasos. Aca estamos calculando todo el episodio y sus rewards
- La expectation (promedio ponderado) es segun la distribucion conjunta de las variables aleatorias: 
    - estados futuros (si entorno es estocastico)
    - acciones futuras (si politica es probabilistica)
    - r podria ser tambien si fuera probabilistica pero ahora asumimos que no

Entonces seria: partimos de un estado, y elegimos una accion. Entonces para ver cual es el valor que tiene tomar esa accion en ese estado, sumamos los rewards descontados para todo el episodio. Y promediamos por todos los posibles episodios que pueden haber, segun sus probabilidades. Como se promedia eso? 
- Cuando elegimos una accion a en el estado s, eso nos puede llevar al estado s'_1 (0.3) o al estado s''_1 (0.7)... 
- Para cada branch tomamos una nueva decision basada en la politica. Esas acciones tambien pueden ser probabilistica, por lo que si tenemos dos acciones, ahora vamos a tener 4 branches: (s'_1, a'_1), (s'_1, a''_1), (s''_1, a'_1), (s''_1, a''_1). 
- Despues, para cada una de esas acciones hay dos posibles estados, entonces tenemos 8 branches... y asi. 

Entonces, teoricamente deberiamos promediar los rewards que va recopilando cada trayectoria, por TODAS esas trayectorias/branches, ponderadas por su probabilidad

**Q\*** (Q-star)

Lo anterior es con una politica, pero nosotros queremos definir teoricamente esto con **la mejor politica posible**. Q* es el valor de tomar una accion a en un estado s, si desde ahora en adelante elegimos siempre las mejores acciones.

Bellman Optima 

$$Q^*(s, a) = R(s, a) + \gamma \, \mathbb{E}_{s' \sim p_S(\cdot|s,a)} \left[ \max_{a'} Q^*(s', a') \right]$$

Seguimos en lo mismo: estamos en s y tenemos que elegir un a... Queremos saber teoricamente cuanto vale cada accion (la elegimos nosotros). El (mejor) valor de hacer a en s es: el reward inmediato mas el valor esperado del mejor movimiento en el proximo estado.
- El promedio es sobre los estados (entorno estocastico).
- Las acciones futuras ya no son aleatorias: se asume que se elige la mejor siempre (max a')

**TD update rule (Q-learning)**

Estimacion muestral en practica: Como no podemos calcular la expectativa, usamos TD learning:

$$Q(s, a) \leftarrow Q(s, a) + \eta \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

Con esto voy aprendiendo mi estimacion de Q(s,a)

**Politica estrategia**

- **Greedy**: Como ya tengo Q(s,a), puedo probar todas las a y elegir la que mayor valor tenga, siempre
$$a = \arg\max_{a'} Q(s, a')$$
(aca a' no es algo del futuro sino que "recorre" todas las posibles a que hay en ese momento y elige la mejor)
- **eps-greedy** (Con exploracion): en vez de elegir siempre la mejor, la elegimos con 1-eps de prob (explotar). Entonces con eps, elegimos una accion al azar (explorar)

Resumen:
- Q∗ mide el valor de las acciones si después siempre jugás óptimo.
- La ecuación de Bellman te dice cómo se relaciona el valor actual con el valor futuro.
- Q-learning aprende esa relación usando muestras reales y un update TD.
- La política resultante es simplemente “elegí la acción con mayor Q”.

**Q-learning aprende una tabla de valores Q(s,a) y despues elige la mejor accion de una lista (argmax). Funciona ok para pocas acciones y discretas. Si es continua no podemos usar argmax sobre infinitas acciones**.

---

### Policy-based RL

En vez de aprender valores, **ahora aprendemos directamente la politica como modelo parametrico pi(a|s). Una funcion que te da directamente la accion o su distribucion** (como una red neuronal).

Objetivo (valor esperado total que el agente obtiene usando esa politica):

$$J(\pi_\theta) = \mathbb{E}_{p(s_0)} \left[ V^{\pi_\theta}(s_0) \right]$$

**policy search**
- Queremos ajustar los parametros theta para maximizar el reward esperado
- La expectation es sobre toda la distribucion de trayectorias que puede generar la politica:

$$J(\pi_\theta) = \mathbb{E}_{p(s_0)} \left[ V^{\pi_\theta}(s_0) \right] = \mathbb{E}_{p(s_0)} \left[ \mathbb{E}_{\pi_\theta, p_S} \left[ \sum_{t=0}^{\infty} \gamma^t r_t \mid s_0 \right] \right]$$

- p(s_0): la distribucion de estados iniciales (puede ser aleatorio)
- pi(a_t|s_t): la politica: prob de elegir cada accion dado el estado
- p_s(s_t+1|s_t,a_t): la dinamica del entorno: prob de a que estado pasamos dado una accion y un estado actual
- La prob de una trayectoria completa puntual es:

$$p_\theta(\tau) = p(s_0) \prod_{t=0}^{\infty} \pi_\theta(a_t|s_t) \, p_S(s_{t+1}|s_t, a_t)$$

El objetivo dice: promedia todos los rewards acumulados de todas las trayectorias posibles, ponderados por cuan probables son esas trayectorias segun pi. 
- pi define una distribucion sobre trayectorias posibles
- cambiando theta, cambiamos esa distribucion de trayectorias

**Policy gradient**

Si J(pi) es diferenciable respecto de theta, podemos usar SGD (o ascent). Es hacer backprop pero a traves del comportamiento del agente.

$$\theta \leftarrow \theta + \alpha \nabla_\theta J(\pi_\theta)$$

- Como no usamos argmax, podemos usarlo en **espacios de accion continuos**
- **PROBLEMA: alta varianza en el gradiente** cuando usamos una muestra de trayectorias -> actualizaciones muy ruidosas del gradiente. 
    - updates cambian mucho de tamaño y direccion de un batch al otro
    - rewards muy dispersos: a veces 0 a veces 100, etc
    - provoca que el gradiente estimado con las muestras es inestable.


---

**Policy-gradient: Derivacion**: Si tenemos la definicion de J(pi) (la pasamos a formato expectation por trayectoria asi es mas facil):

$$J(\pi_\theta) = \mathbb{E}_{s_0}\left[ V^{\pi_\theta}(s_0) \right] = \mathbb{E}_{\tau \sim p_\theta(\tau)} \left[ R(\tau) \right] = \mathbb{E}_{\tau \sim p_\theta(\tau)} \left[ \sum_{t=0}^{\infty} \gamma^t r_t \right]$$

La expectation ampliada es simplemente el reward promedio, ponderado por la prob de cada trayectoria:

$$J(\pi_\theta) = \int R(\tau) \, p_\theta(\tau) \, d\tau$$

Queremos el gradiente de J respecto a theta (como cambia el reward esperado si modificamos los parametros theta)
- Cuando derivamos, lo unico que depende de theta es la distribucion $p_\theta(\tau)$
- Los rewards son numeros fijos, una vez que conocemos la trayectoria

Entonces tenemos gradiente teorico: 
$$\nabla_\theta J = \int R(\tau) \nabla_\theta p_\theta(\tau) \, d\tau$$

- Esto es: como cambia la probabilidad de cada trayectoria, cuando ajustamos theta.
- Como no conocemos todas las trayectorias posibles, no lo podemos integrar (hay infinitas combinaciones de estados, acciones y recompensas). Usamos una **estimacion por muestreo (monte carlo)**: 

Score function estimator (log-derivative trick). Es una identidad util:

$$\nabla_\theta p_\theta(\tau) = p_\theta(\tau) \nabla_\theta \log p_\theta(\tau)$$
- Derivar un log y multiplicar por la original recupera la derivada. Ejemplo:
$$\nabla_\theta p = p \nabla_\theta \log p$$

Entonces nos queda:

$$
\nabla_\theta J
= \int R(\tau)\, p_\theta(\tau)\, \nabla_\theta \log p_\theta(\tau)\, d\tau
$$

Y eso lo podemos escribir como una **expectativa**:

$$
\nabla_\theta J
= \mathbb{E}_{\tau \sim p_\theta(\tau)} \left[ R(\tau)\, \nabla_\theta \log p_\theta(\tau) \right]
$$

*Ya no tenemos una integral imposible — tenemos una expectativa que se puede estimar muestreando trayectorias $\tau$.*

Paso 4. Expandimos $ \log p_\theta(\tau) $

Recordemos que:

$$
p_\theta(\tau)
= p(s_0)\, \prod_{t=0}^{T-1} \pi_\theta(a_t|s_t)\, p_S(s_{t+1}|s_t, a_t)
$$

Tomando el log:

$$
\log p_\theta(\tau)
= \log p(s_0) + \sum_{t=0}^{T-1} \left[ \log \pi_\theta(a_t|s_t) + \log p_S(s_{t+1}|s_t, a_t) \right]
$$

Pero **solo** la política depende de $\theta$, no el entorno.
Así que:

$$
\nabla_\theta \log p_\theta(\tau)
= \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)
$$

Paso 5. Sustituimos nuevamente

$$
\nabla_\theta J
= \mathbb{E}_{\tau \sim p_\theta(\tau)} \left[
R(\tau) \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)
\right]
$$

Y **esa es la Policy Gradient Theorem**, en su forma más directa
(la versión Monte Carlo del algoritmo REINFORCE).

Intuición clara

* $R(\tau)$: mide *qué tan buena fue* una trayectoria completa.
* $\nabla_\theta \log \pi_\theta(a_t|s_t)$: mide *cómo cambiar $\theta$* para hacer esa acción más o menos probable.

Entonces el producto:

$$
R(\tau)\, \nabla_\theta \log \pi_\theta(a_t|s_t)
$$

significa:

> “si esta trayectoria fue buena ($R$ alto), aumentá la probabilidad de las acciones que tomaste;
> si fue mala, reducilas”.

Y al tomar la expectativa, el modelo aprende la dirección promedio que aumenta el reward esperado.


Paso 6. Forma estimable

Como no podemos calcular la expectativa analíticamente, la estimamos por Monte Carlo:

$$
\hat{g}
= \frac{1}{N} \sum_{i=1}^{N} \sum_{t=0}^{T-1}
\nabla_\theta \log \pi_\theta(a_t^{(i)}|s_t^{(i)})\, R(\tau^{(i)})
$$

Cada trayectoria ($\tau^{(i)}$) viene de ejecutar la política actual en el entorno.

Y acá aparece la **alta varianza** Porque:
* cada (R(\tau)) puede variar mucho entre trayectorias,
* y estamos multiplicando todo el gradiente por ese número grande (o chico).


---

**Reducir la varianza: actor-critic**

Como estamos actualizando la politica (las acciones) segun si dieron buenos resultados o malos: R(tau), y eso tiene mucha varianza (por ej el R puede dar muy distinto con acciones muy parecidas), entonces ese ruido hace que el gradiente cambie de direccion todo el tiempo y sea lento e inestable. Por azar podes caer en lugares buenos y te da reward altisimo (o todo al reves), etc.

**Restar Baseline**: entonces pensamos, en vez de usar el reward puro, podemos ver si nos fue mejor o peor **de lo que esperaba**. Entonces, en lugar de multiplicar R(tau), multiplicamos por:

$$R(\tau) - \text{baseline}$$

- Si me fue mejor que el baseline -> refuerzo
- Si me fue peor -> castigo
- Si fue igual -> no hago nada

Restar el baseline es como centrar los datos: te quedás con cuánto mejor o peor fue cada muestra que el promedio.
Eso hace que el ruido alrededor del valor esperado se reduzca y que el gradiente “mire” más claramente hacia la dirección correcta.

**Value function V(s) como baseline**

Value function es: **cuanto espero ganar desde este estado, siguiendo mi politica**. Es el promedio (expectation) de todas las trayectorias posibles (aleatoriedad en policy y estados del entorno) siguiendo mi politica, desde el estado actual. 
- NO SE CONOCE porque imposible saber todos los estados posibles
- SE ESTIMA o aprende

$$R(\tau) - V_w(s_t)$$

- w: son los parametros del critic, el modelo que estima la value function (theta son los params de la policy)

La resta se conoce como **Advantage**.
- Tambien se podria aprender directamente el advantage, en vez del value function


Este es el modelo **actor-critic**:
- Actor: la policy que va actualizando theta para hacer mas probables las acciones con advantage positivo
- Critic: el estimador V(s) o A(s,a) que evalua como le esta yendo al actor. 
    - Se entrena usando TD learning para predecir el retorno esperado
Entonces:
- $\pi_\theta(a \mid s)$ → política (actor), con parámetros $\theta$.
- $V_w(s)$ o $A_w(s, a)$ → estimador del valor (critic), con parámetros $w$.
- Y ambos se entrenan simultáneamente, pero con objetivos distintos.

**Aprender $V_w(s)$ con TD learning**

TD learning es **aprender a predecir el futuro, sin esperar a que el futuro llegue por completo**.
- Tenemos un V(s) que es lo que pensamos nosotros del valor de s
- Damos un paso y vemos que s quizas vale mas (tenemos mas info ahora, porque tenemos r y V(s'))
- Actualizamos V(s) incorporando esta info

Lo que suponiamos que s valia:
$$V_w(s_t)$$
- Es una red neuronal o funcion parametrica, con parametros w, que estima V(s)

Lo que sabemos despues de dar un paso:
$$r_t + \gamma V_w(s_{t+1})$$
La diferencia entre lo que pensabamos y lo que ahora vemos (TD error):
$$\delta_w = (r_t + \gamma V_w(s_{t+1})) - V_w(s_t)$$
Entonces, conociendo eso, **podemos actualizar V(s) despues de dar un paso en el futuro (temporal difference)**:
$$V_w(s_t) \leftarrow V_w(s_t) + \eta \left[ r_t + \gamma V_w(s_{t+1}) - V_w(s_t) \right]$$
Esto es conceptual, como si fueran valores en una tabla (TD learning tabular). Aca trabajamos con **redes neuronales**

**TD learning con funcion aproximadora**: El valor depende de w, entonces no se puede actualizar el valor directamente sino que tenemos que ajustar los parametros de la red.

Usamos el **TD error (delta) como target**, y la regla de actualizacion queda:

$$w \leftarrow w + \eta \, \delta_t \, \nabla_w V_w(s_t)$$
$$w \leftarrow w + \eta \, \left[ r_t + \gamma V_w(s_{t+1}) - V_w(s_t) \right] \, \nabla_w V_w(s_t)$$

- No derivamos lo que hay dentro de delta_t $V_w(s_{t+1})$ porque lo consideramos parte del target, delta_t es un numero.

---

### Model-based RL

Tanto Value-based (Q-learning) como policy-based (policy gradient) son muy ineficientes porque tienen que interactuar mucho con el entorno (el entorno es real) para poder aprender algo. Cada interaccion real con un entorno de verdad cuesta tiempo o dinero.

La idea aca es **aprender un modelo del mundo / entorno para poder SIMULAR internamente, sin gastar interacciones reales**.
- El agente aprende un modelo del entorno (un simulador mental). Tiene:
    - Funcion de transicion: Si estoy en un estado y hago esta accion, a donde voy despues?
    - Funcion de recompensa: Si hago esta accion en este estado, que recomepensa espero obtener?

La idea es **predecir resultados sin interactuar realmente**.

Una vez que se tiene el modelo del entorno, podemos PLANEAR:
- Imaginar varios futuros posibles.
- Calcular qué acciones llevarían a mejores recompensas.
- Ajustar mi política sin tocar el entorno real.
Maneras:
- Approximate Dynamic Programming: resolver el problema usando ecuaciones tipo Bellman, pero dentro del modelo aprendido.
- **Lookahead search**: probar mentalmente distintas secuencias de acciones (como hace **AlphaZero con MCTS**).

**Aprendizaje**

Se aprende todo junto, iterativamente:
- El agente actúa un poco en el entorno real → recolecta datos.
- Usa esos datos para mejorar su modelo del mundo.
- Usa ese modelo para entrenar su política internamente (simulando).
- Usa esa política mejorada para actuar mejor y recolectar mejores datos.
- Se repite el ciclo

**World model** es definido con varios componentes:
- Modelo de transicion $p_S(s'|s,a)$: predice el proximo estado del mundo dado un estado y una accion
- Modelo de observacion $p_O(o'|s')$: predice que observacion voy a ver, dado un nuevo estado (si el agente no ve todo)
- Modelo de recompensa $p_R(r|s,a)$: predice la recompensa esperada por esa accion en ese estado

Hoy en dia tambien se usa en sentido mas amplio, como el modelo interno del entorno que el agente usa para imaginar, predecir o planificar mentalmente.
- En **Dreamer** (de DeepMind), el world model es una red neuronal que codifica el entorno entero en un estado latente z_t
- Aprende a predecir $z_{t+1}$, $r_t$ y a reconstruir observaciones $o_t$
- Despues entrena politica dentro de ese espacio imaginado, sin usar entorno real


**Dreamer** (DeepMind, 2020) busca aprender y actuar en entornos visuales complejos (tipo Atari o simuladores 3D),
sin necesitar millones de pasos en el entorno real.
- Aprende un world model en espacio latente
    - Encoder: observa imagenes reales del entorno y larga un vector latente que resume lo importante
    - Transition model: toma el estado latente y una accion y devuelve el siguiente estado latente imaginado (RNN)
    - Decoder: toma un estado latente y devuelve la imagen reconstruida (sirve para entrenar el world model, asegurando que su representacion tenga sentido)
    - Reward model: toma el estado latente y una accion y devuelve el reward esperado
    - Se entrena con datos reales (imagenes, acciones, rewards)
- Actor: recibe el estado latente actual y predice que accion tomar. Se entrena dentro del mundo imaginario, ajustando la politica para maximizar los rewards imaginados. Se entrena con policy based methods, como REINFORCE
- Critic: estima que tan bueno es estar en un estado latente. Devuelve el valor esperado V. Se entrena con TD learning, usando trayectorias imaginadas por el world model
(dreamer no planea en tiempo real con planificacion (como alphago con MCTS por ej) sino que imagina para entrenar)

### Repaso

**Reward function ($R(s, a)$)**

La reward function describe el “feedback” que el entorno devuelve al agente. Puede entenderse en dos niveles: inmediato y de trayectoria.

**Reward inmediato**  
El reward inmediato es lo que el entorno entrega en el momento en que el agente toma una acción.

$$
r_t = R(s_t, a_t)
$$

Es una función definida por el entorno, que puede ser determinista o estocástica. Refleja directamente qué tan buena fue una acción en un estado.  
Ejemplo: en un videojuego, +10 puntos por agarrar una moneda, −5 si te caés.  
En este caso no hay expectation, porque el reward se observa directamente del entorno. Si el entorno es estocástico, lo que se observa es una muestra de una distribución $p(r_t \mid s_t, a_t)$.

**Reward de trayectoria**  
Cuando se evalúa una trayectoria completa, el reward se define como la suma de todos los rewards inmediatos descontados:

$$
R(\tau) = \sum_{t=0}^{T-1} \gamma^t r_t
$$

Esta es una cantidad que resume toda la experiencia del episodio, y se utiliza para evaluar qué tan buena fue la política completa que generó esa trayectoria.  
No se toma ninguna expectation aquí: este es el total observado en una trayectoria específica.

**Cuándo se conoce:**  
El reward inmediato se observa en cada paso. El reward total de una trayectoria se conoce recién al finalizar el episodio.  
**Tipo:** observado directamente del entorno (no se estima).

---

**Return ($G_t$)**

El return representa la suma de todos los rewards futuros a partir del tiempo $t$:

$$
G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \ldots
$$

Incluye tanto el reward actual como los futuros, ponderados por el discount factor $\gamma$.

El return se usa para medir “cuánto valió” una secuencia completa de decisiones desde un punto específico.  
No hay expectation: $G_t$ es el return realmente observado en una trayectoria particular.

**Cuándo se conoce:** al final del episodio, cuando ya se observaron todos los rewards.  
**Tipo:** valor observado (sirve como señal para entrenar modelos que predicen el valor).

---

**Value function ($V^\pi(s)$)**

La value function mide el valor esperado del return total si el agente sigue la política $\pi$ desde un estado $s$:

$$
V^\pi(s) = \mathbb{E}_{\tau \sim p_\pi(\tau \mid s)} [G_t]
$$

Es una expectation (promedio ponderado) sobre todas las trayectorias futuras posibles a partir de $s$, donde la probabilidad de cada trayectoria está dada por:

* la política $\pi$ (que determina las acciones),
* y las dinámicas del entorno (que determinan los siguientes estados y rewards).

En palabras simples: $V^\pi(s)$ es el promedio de todos los posibles retornos futuros que se podrían obtener desde el estado $s$, ponderados por qué tan probables son bajo la política actual.

**Cuándo se conoce:** nunca exactamente, porque depende de todas las posibles trayectorias futuras.  
**Tipo:** cantidad teórica, se **estima** en la práctica (por ejemplo, con Monte Carlo o TD learning) a partir de experiencias del entorno.

**Intuición:** “Si estoy en este estado y sigo actuando según mi política, ¿cuánto espero ganar en promedio desde ahora hasta el final?”

---

**Q function ($Q^\pi(s,a)$)**

La Q-function es similar a la value function, pero además especifica cuál es la acción inicial:

$$
Q^\pi(s,a) = \mathbb{E}_{\tau \sim p_\pi(\tau \mid s,a)} [G_t]
$$

La expectation se toma sobre todas las trayectorias futuras que comienzan en el par $(s,a)$, ponderadas por:

* la política $\pi$ (para las acciones siguientes),
* y las transiciones del entorno (para los próximos estados).

En otras palabras, $Q(s,a)$ responde: “si en este estado hago esta acción, y después sigo actuando como suelo hacerlo, ¿cuánto espero ganar en promedio?”.

**Cuándo se conoce:** tampoco se conoce exactamente, se estima por experiencia.  
**Tipo:** cantidad teórica, se **estima** mediante aprendizaje (por ejemplo, TD o Q-learning).

**Intuición:** permite comparar acciones en un mismo estado. La política óptima es la que elige la acción con el $Q$ más alto.

---

**Objective function ($J(\pi)$)**

La función objetivo mide qué tan buena es una política completa en promedio sobre todos los posibles estados iniciales:

$$
J(\pi) = \mathbb{E}_{s_0 \sim p_0}[V^\pi(s_0)] = \mathbb{E}_{\tau \sim p_\pi(\tau)} [R(\tau)]
$$

La expectation se toma sobre:

* la distribución inicial de estados $p_0(s_0)$,
* y todas las trayectorias posibles generadas por la política $\pi$ y el entorno.

Es la medida global de rendimiento: el reward esperado total que obtiene la política en promedio.

**Cuándo se conoce:** no se conoce exactamente; se **estima** empíricamente promediando los returns observados en episodios ejecutados con la política actual.  
**Tipo:** estimado empíricamente (mide el desempeño global).

**Intuición:** “¿Qué tan buena es mi política en promedio, considerando todos los posibles comienzos?”


# RL para LLMs

- Agent/policy model: Estrategia para tomar decisiones. Es el LLM
- Environment: provee feedback al agente. Puede ser muchas cosas aca
- Action: Pueden ser PALABRAS, RESPUESTA ENTERA, OUTCOME FINAL, etc
- Reward: Feedback que le da el env al agent. Numero

---

**RLHF**

- Preferencias humanas: a partir de varias respuestas, las ranqueamos
- Reward model: construimos un reward model basado en esas preferencias. Ahora tenemos un modelo que, dado una respuesta, estima el puntaje que le darian humanos
- Fine-tune el LLM con RL: Ahora el reward model nos da el feedback. Entonces:
    - El agente genera respuestas
    - El reward model las scorea
    - Ajustamos los weights para que tienda a producir outputs que sean bien puntuados


---

**RL vs SL**

Principalmente cambia la fuente de la señal (reward). Qué optimizas realmente
- SL: Minimizar divergencia respecto a la distribución objetivo (imitar). Tiene un reward **denso** porque sabes la respuesta target para cada token.
- RL: Maximizar retorno esperado bajo dinámicas del entorno (elegir). La señal es escasa y esta diferida en el tiempo. Se requiere **exploracion** y credito temporal. 

Por eso se dice que SFT memorizes y RL generalizes https://arxiv.org/abs/2501.17161

---



# Policy optimization (gradient-based)

Tecnicas para optimizar el LLM / policy model para que mejore las predicciones.

**Intro**:

En SL tenemos una funcion de perdida (loss) y lo que queremos es minimizar esa loss. Entonces dado un batch, calculamos la loss (un numero final) basado en muchos errores (1 o varios para cada elemento del batch), los promediamos y nos queda ese numero final. Ahi, buscamos el gradiente (el conjunto de derivadas parciales de cada uno de los parametros aprendibles) que hagan que mas baje la loss en ese punto. Y actualizamos los valores de los parametros en la direccion y fuerza de esas derivadas y agregandole otros empujones.

En RL, lo que tratamos de hacer es subir o bajar la probabilidad de los tokens segun si aparecen en la secuencia que genero un buen o mal resultado (bien o mal comparado con un baseline).


**Pasos general (on-policy)**:
- Se hacen rollouts (el modelo genera respuestas completas)
- Se evalua cada respuesta y se obtiene score o reward
- El score se convierte en ventaje (advantage): cuanto mejor o peor esta respuesta fue a lo esperado o tipico (por ej del grupo o de algo de referencia).
    - Si tengo solo reward final, el advantage se aplica a todos los tokens de la secuencia de la misma manera (credito global)
    - Si tengo verificadores por paso, se reparten ventajas distintas por token
- Distribuyo la ventaja para cada token:
    - Si es positiva, subo su log-prob (empujo logits para arriba).
    - Si es negativa, la bajo.
    - Indirectamnete afecto a las otras (porque en probs la suma = 1)
- Regularizo para no irme tan lejos de una distribucion anterior (basada en una politica de referencia) para evitar colapso.

**Caso general**
- Nosotros tenemos una red y para un estado (secuenci), produce logits (next token / action)
- Los pasamos por softmax y tenemos probs y de esos tomamos el log, por lo que trabajamos con log probs. 
- Para el token elegido en un paso t, vemos el log prob (solo tenemos un log prob en ese paso)
- Si tomamos el gradiente de ese log prob, estamos viendo un conjunto de derivadas parciales del logprob con respecto a todos los parametros del modelo. Esto indica la direccion de MAXIMA SUBIDA -> o sea, si tomamos un paso en esa direccion, aumentamos la logprob de ese token para la prox.
- Si hacemos lo mismo para cada token de la secuencia de pasos generados, y sumamos los gradientes (sumamos las derivadas parciales (o sea, para un parametro miramos las derivadas parciales por cada token en la secuencia generada y los sumamos)), nos queda un gradiente unico que apunta a maximizar todos los tokens en la secuencia generada. 
- Despues vemos el reward para esa secuencia y lo comparamos con el baseline (R-b). Si el baseline es 0.5 y nuestro reward es 0.2, el advantage da -0.3. Eso lo usamos para multiplicar, lo que nos cambia la direccion del gradiente si el advantage es negativo y tambien nos da una magnitud de cambio (escala el tamaño del paso junto con el lr). O sea que si es negativo, en vez de apuntar a aumentar la prob, apunta a disminuir la prob de los tokens.
- Para no favorecer secuencias largas, a veces se promedia la suma de logprobs y se suele agregar entropia (para mantener diversidad) y KL a un modelo de referencia para regularizar.


## REINFORCE

Loss function: 

$$\text L_\theta = - (R - b) \sum_t \log \pi_\theta(a_t \mid s_t)$$

- Partimos de Maximum Likelihood: queremos que el modelo asigne alta probabilidad a lo que efectivamente ocurrió (los datos reales).
- Para secuencias, eso se traduce en un producto de probabilidades de cada token de la secuencia.
- Como trabajar con productos es numéricamente inestable (tienden a 0), usamos logaritmo: eso convierte el producto en suma.
- Advantage 𝐴=𝑅−𝑏: En RL, no todas las muestras valen lo mismo: algunas tuvieron mejor reward. Por eso ponderamos la suma con el advantage. El Advantage puede ser por paso (cuando haya reward por paso u otra tecnica) o general para la secuencia (para una secuencia particular, con respecto a otras (baseline)). Se puede poner dentro de la suma.

$$\text L_\theta = - \sum_t A_t \cdot \log \pi_\theta(a_t \mid s_t)$$

- La loss depende de θ. Como regla de gradient descent tenemos: $\theta \leftarrow \theta - a \cdot \nabla_\theta L$.
- Queremos derivar la loss respecto a θ, modificando los pesos (que es lo unico que podemos mover) para que la loss sea mas chica.
- Tenemos la politica (que es el LLM), que esto lo podemos mover, porque depende de θ. Entonces lo queremos derivar: $\nabla_\theta \log \pi_\theta(a_t \mid s_t)$
- El gradiente de una funcion siempre nos dice el steepest ascent de la funcion con respecto a un parametro.
- Si tenemos una Loss, vamos a querer derivarla respecto a los parametros del modelo. Entonces buscamos en la formula aquello que dependa de los parametros del modelo (pi)

Entonces, el gradiente para la secuencia entera seria:

$$\nabla_\theta L_\theta = - \sum_t A_t \cdot \nabla_\theta \log \pi_\theta(a_t \mid s_t)$$

Este es el vector del gradiente, que apunta a donde mas crece L.

Y en cuanto a gradient descent tenemos que, en cada optimizacion, theta cambia asi:

$$\Delta \theta = - a \cdot \nabla_\theta L(\theta)$$

O sea: 

$$\theta \leftarrow \theta - a \cdot \nabla_\theta L(\theta)$$

In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

torch.manual_seed(0)

vocab_size=5
T=3
lr=0.5

# Model: linear map from one-hot(state) to vocab logits (bias-free for clarity).
model = nn.Linear(T, vocab_size, bias=False)
# Initialize weights small & reproducible
with torch.no_grad():
    model.weight.copy_(0.1 * torch.randn(vocab_size, T))

opt = optim.SGD(model.parameters(), lr=lr)

log_softmax = nn.LogSoftmax(dim=-1)

model.weight

Parameter containing:
tensor([[ 0.0604,  0.0811, -0.0045],
        [ 0.0880,  0.1048, -0.0045],
        [-0.0723,  0.2866, -0.0566],
        [ 0.0160, -0.0025,  0.1074],
        [ 0.2263, -0.0918, -0.0225]], requires_grad=True)

In [31]:
# Inputs: one-hot vectors for each time step 0..T-1
I = torch.eye(T)  # shape (T, T)
I

tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]])

In [32]:
# CASO 1 (GOOD)
chosen_tokens = [2, 1, 4]
advantage = 0.4

# --- Forward BEFORE update: record chosen tokens' log-probs and probs ---
with torch.no_grad():
    before = []
    for t in range(T):
        logits_t = model(I[t])          # (vocab,)
        logp_t  = log_softmax(logits_t) # (vocab,)
        tok = chosen_tokens[t]
        before.append((tok, float(logp_t[tok]), float(logp_t.exp()[tok])))
before

[(2, -1.750232458114624, 0.17373354732990265),
 (1, -1.5883560180664062, 0.20426113903522491),
 (4, -1.6373724937438965, 0.19449038803577423)]

In [33]:
# --- Compute REINFORCE loss:  L = -A * sum_t log pi(a_t | s_t) ---
total_logprob = 0.0
for t in range(T):
    # sumamos todos los logprobs de los tokens elegidos
    logits_t = model(I[t])          # (vocab,)
    logp_t  = log_softmax(logits_t) # (vocab,)
    tok = chosen_tokens[t]
    total_logprob = total_logprob + logp_t[tok]
    print(f't={t}: token {tok} logprob={logp_t[tok]} total_logprob={total_logprob}')
print(f'-advantage={-advantage}\n-advantage * total_logprob = {-advantage} - {total_logprob}')
loss = - advantage * total_logprob
loss

t=0: token 2 logprob=-1.750232458114624 total_logprob=-1.750232458114624
t=1: token 1 logprob=-1.5883560180664062 total_logprob=-3.3385884761810303
t=2: token 4 logprob=-1.6373724937438965 total_logprob=-4.975960731506348
-advantage=-0.4
-advantage * total_logprob = -0.4 - -4.975960731506348


tensor(1.9904, grad_fn=<MulBackward0>)

In [34]:
# Backprop + step
opt.zero_grad()
loss.backward()
opt.step()

model.weight

Parameter containing:
tensor([[ 0.0207,  0.0412, -0.0441],
        [ 0.0472,  0.2640, -0.0441],
        [ 0.0930,  0.2376, -0.0941],
        [-0.0219, -0.0392,  0.0631],
        [ 0.1794, -0.1253,  0.1386]], requires_grad=True)

In [37]:
# --- Forward AFTER update: record chosen tokens' log-probs and probs ---
with torch.no_grad():
    after = []
    for t in range(T):
        logits_t = model(I[t])          # (vocab,)
        logp_t  = log_softmax(logits_t) # (vocab,)
        tok = chosen_tokens[t]
        after.append((tok, float(logp_t[tok]), float(logp_t.exp()[tok])))

print("Per-step chosen token probabilities (before -> after):")
for t, ((tok_b, logpb, pb), (tok_a, logpa, pa)) in enumerate(zip(before, after)):
    assert tok_b == tok_a
    arrow = "↑" if pa > pb else ("↓" if pa < pb else "→")
    print(f"  t={t}: token={tok_b}   P_before={pb:.4f} -> P_after={pa:.4f}   ({arrow})")

Per-step chosen token probabilities (before -> after):
  t=0: token=2   P_before=0.1737 -> P_after=0.2055   (↑)
  t=1: token=1   P_before=0.2043 -> P_after=0.2386   (↑)
  t=2: token=4   P_before=0.1945 -> P_after=0.2280   (↑)


## PPO

Limita cuanto se puede cambiar la distribucion para que no hayan pasos demasiado grandes y colapse.

Hay un ratio por token

$$r_t = \frac{\pi_{\text{nueva}}(a_t \mid s_t)}{\pi_{\text{vieja}}(a_t \mid s_t)}$$

Que te dice cuanto cambiaste la prob para un token. Si es 1.3, subiste 30% la probabilidad para ese token. Si es 0.7 bajaste 30% la prob para ese token.

- **Clipping**: Si el paso te lleva a un r_t demasiado grande > 1 + eps, entonces recorto el beneficio
- **KL divergence a un modelo referencia**: Penaliza alejarte del modelo base de referencia (el original) en promedio.
- **Ventaja por paso** (opcional): en vez de tener un unico A global, lo hacemos por paso, con V(s_t) para asignar mejor el credito y premiar a los tokens clave. Si no esta, se puede usar como baseline la media del batch.
- **Entropia**: bonus que mantiene la exploracion y evita colapso de entropia (que se vuelva modo unico)

**PPO Loss**:

$$L = \mathbb{E}\left[ \min\left(r_t A_t, \; \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t \right) \right] - \beta \, \mathrm{KL}(\pi_{\text{new}} \| \pi_{\text{ref}}) + \alpha \, \text{Entropía}$$

Tres grandes partes:

- Politica (clipped surrogate objective) -> $\mathbb{E}\left[ \min\left(r_t A_t, \; \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t \right) \right]$
- KL -> $- \beta \, \mathrm{KL}(\pi_{\text{new}} \| \pi_{\text{ref}})$
- Entropia -> $\alpha \, \text{Entropía}$


**Gradiente**:

- Lo mismo que REINFORCE, pensamos: Queremos minimizar esta loss. Como hacemos? Que depende de nosotros? los theta. Y donde estan, en pi_new. Todo lo demas es constante porque no depende de nosotros.
- Donde esta pi_new? En el ratio: 
$$r_t = \frac{\pi_{\text{new}}(a_t \mid s_t)}{\pi_{\text{old}}(a_t \mid s_t)} = \exp\left( \log \pi_{\text{new}} - \log \pi_{\text{old}} \right)$$
- Por eso al derivar, todo fluye por pi_new.
- PROPIEDAD por la exp: la derivada del ratio es el propio ratio multiplicado por la derivada del log-prob nuevo:

$$\nabla_\theta r_t = r_t \cdot \nabla_\theta \log \pi_{\text{new}}(a_t \mid s_t)$$
 
$$\nabla_\theta (\exp\left( \log \pi_{\text{new}} - \log \pi_{\text{old}} \right)) = (\exp\left( \log \pi_{\text{new}} - \log \pi_{\text{old}} \right)) \cdot \nabla_\theta \log \pi_{\text{new}}$$

Esto se da porque cuando tenemos una funcion exp:

$y = \exp(g(\theta))$

Al derivar exp, te devuelve exp otra vez, y despues multiplicas por la derivada de lo de adentro (**chain rule**):

$\nabla_\theta y = y \cdot \nabla_\theta g(\theta)$

- pi_old no esta porque no depende de theta, es constante -> $\exp(\log \pi_{\text{new}} - \text constante) $
- Entonces, cuando r_t = 1 (al principio cuando las dos policies son iguales), entonces el gradiente es igual a REINFORCE!

**Politica (clipped surrogate objective)**:

En REINFORCE tenemos A_t que es un empuje al gradiente: $A_t \cdot \nabla_\theta \pi$:
- Grad de log pi: es la flecha (vector de cambios) que te dice como mover los pesos para aumentar el valor de la funcion.
- A: Empuja a favor de subir o bajar la prob, segun el reward. Le puede cambiar el signo (si fue malo) y cambiarle intensidad.
    - Si A > 0: queremos subir la prob
    - Si A < 0: queremos bajar la prob

**Ratio pi_new/pi_old: Importance sampling**

Por que multiplicamos por el ratio? IMPORTANCE SAMPLING (es una correccion estadistica).

Las trayectorias se recolectaron con pi_old (porque es caro volver a samplear), pero querés optimizar como si vinieran de pi_new. Estamos trabajando con samples **out of distribution (off-policy)** -> Si bien PPO medio que se clasifica como on-policy en general, esta parte es off-policy.

El verdadero gradiente que quiero estimar:
$$\mathbb{E}_{\tau \sim \pi_{\text{new}}} \left[ \sum_{t} \nabla_\theta \log \pi_{\text{new}}(a_t \mid s_t) A_t \right]$$
Esto es lo ideal. Es la esperanza bajo la POLITICA NUEVA.

Lo que tenemos en la practica:
- Los datos vienen de pi_old. O sea, que si queremos el expectation (dandole mas o menos peso)

Lo que queremos es CORREGIR LA ESPERANZA.
- Un promedio comun es: 1/N sum(x_i) -> cada ejemplo pesa igual. Esto es igual a: sum(x_i * 1/N)
- Un promedio ponderado es: sum(x_i*w_i) / sum(w_i) -> para que sea promedio (suma de pesos = 1). NORMALIZADO.
- En IMPORTANCE SAMPLING queremos el promedio COMO SI viniera de otra distribucion. 
    - Se suele usar version no normalizada por propiedad estadistica: E_q(r(x)*f(x)) = E_p(f(x)) cuando el ratio esta calculado exacto p(x)/q(x)
    - Si tomamos muestras de q, multiplicamos cada valor por r y hacemos el promedio simple 1/N, ya recuperamos la expectativa bajo p: (no hace falta dividir por sum(w_i)). Quedaria: **1/N sum(w_i * x_i)**
    - Por eso en PPO vemos r dentro de la expectation! Es simplemente la ponderacion de cada 

Por que? Porque si hicieramos 1/N, le estamos dando importancias distintas a trayectorias. Le damos la importancia de pi_old, que fue desde donde sampleamos, cuando en realidad le queremos dar la importancia de pi_new. IS es promediar con pesos w en vez de 1/N, para que tus muestras (que vienen de la distribución vieja) imiten el promedio “bajo la nueva”. Listo. Eso es IS.

En PPO:
- Tenemos dos expectations/promedios: outer (entre ejemplos (trayectorias) del minibatch), inner (entre tokens dentro de trayectoria). Este ultimo a veces se lo pone como suma nada mas.
- Lo ideal seria ponderar por trayectoria completa, o sea en el outer. El tema es que no queda muy bien porque habria que hacer el producto de los ratio...
- Se suele **ponderar por token** para aproximar a esa ponderacion por trayectoria. Por eso es que vemos el ratio pi_new/pi_old dentro del inner expectation.

$$
\mathbb{E}_{\tau \sim \pi_{\text{old}}} \left[ \mathbb{E}_{\text{tokens} \in \tau} \left[
\frac{\pi_{\text{new}}(a \mid s)}{\pi_{\text{old}}(a \mid s)} \nabla_\theta \log \pi_{\text{new}}(a \mid s) A
\right] \right]
$$

O solo con suma, como se suele hacer a veces:

$$
\mathbb{E}_{\tau \sim \pi_{\text{old}}} \left[ \sum_{(s,a) \in \tau} \frac{\pi_{\text{new}}(a \mid s)}{\pi_{\text{old}}(a \mid s)} \nabla_\theta \log \pi_{\text{new}}(a \mid s) A \right]
$$

TODO: 
- https://arxiv.org/pdf/2508.02833v2
- https://www.arxiv.org/pdf/2508.01203 
- https://medium.com/@fellipe_marcellino/off-policy-reinforcement-learning-with-monte-carlo-302807be32db 
- 


Efectos:
- A la vez de empujarlo con A_t, tambien lo empujamos por el ratio.
- Esto es porque vimos que el grad queda: $\nabla_\theta r_t = r_t \cdot \nabla_\theta \log \pi_{\text{new}}$
- Entonces, los empujes PPO quedarian: $A_t \cdot r_t \cdot \nabla_\theta \log \pi_{\text{new}}$
- Entonces, APARTE DEL **A_t**, ahora tambien lo escalamos por **r_t**: 
    - Si r_t == 1: queda igual a REINFORCE
    - Si r_t > 1: la pi_new ya hace mas probable esa accion que pi_old.
    - Si r_t < 1: la pi_new ya hace menos probable esa accion que pi_old.
- EFECTOS DE ESCALAMIENTO DE GRADIENTES (A_t): 
    - Si A > 0, queremos aumentar la prob de esa accion.
    - Si A < 0, queremos bajar la prob de esa accion.



**CLIP en PPO**:
Importance sampling puede dar pesos extremos (alta varianza). PPO mantiene la corrección pero pone guardarraíles: si r_t se va fuera de [1-ϵ, 1+ϵ] en la dirección que “conviene” a A_t, corta el empuje (grad = 0 para ese token en ese minibatch).


---

**FLUJO**

Los policies new NO SAMPLEAN. Lo unico que samplea es el OLD (se va actualizando cada tanto).

- Tenemos un modelo pi_old. Tenemos prompts. Creamos minibatches (por ejemplo 4 prompts en cada minibatch) y tenemos 3 minibatches (12 ejemplos en total). 
- Sampleamos la respuesta usando pi_old y guardamos toda la info (probs de los tokens para CADA step, reward, etc) para cada prompt en todos los minibatches.
- Ahora empieza la primera corrida de varios EPOCH (1 epoch = 1 pasada por todos los minibatches). Esto se va a hacer SIN HACER NINGUN SAMPLEO A MODELOS. Solo con los datos que estan.
- Agarramos el primer minibatch de 4 ejemplos. Para cada ejemplo ya tenemos la secuencia/respuesta originada por el pi_old. Agarramos el primer ejemplo y vamos token por token de la secuencia guardad por el pi_old y vamos comparando los tokens seleccionados con la prob que arroja el modelo (o sea vamos usando el modelo pi_new para ver la prob que arroja para los tokens muestrados originalmente por el pi_old). Obviamente en la primera vuelta va a ser igual o casi igual que el pi_old porque es el mismo modelo, todavia no se actualizo. Usamos la formula de PPO, sumamos para cada token en la secuencia. -> **El advantage despues lo veo bien**
- Lo mismo para cada ejemplo del batch. Y promediamos para cada ejemplo del batch.
- Despues de cada minibatch, backprop y optimizamos!
- Seguimos con los minibatches hasta terminar todos, hasta ahi 1 epoch
- Hacemos los varios epochs lo mismo. OJO, el pi_new va acumulando los cambios PERO NO VA SAMPLEANDO.
- DESPUES DE ESO, volvemos a samplear pero con el pi_new que reemplaza al pi_old.

Si bien no sampleamos nuevas secuencias con los pi_new a medida que van cambiando, los vamos usando activamente para calcular la prob de cada token de la secuencia original no? O sea tienen un uso muy alto. 
A su vez,es como que los cambios en pi_new se van acumulando durante cada minibatch y eso por varios epochs no? O sea, acumulamos cambios y despues de varios epochs ahi lo usamos para generar una respuesta nueva?


---

**Advantage**

Se hace con actor-critic.

aunque en un episodio ves un solo reward, el crítico se entrena en muchos episodios distintos, y aprende a estimar cuánto se espera desde cada paso en promedio. Esa expectativa por estado es lo que te permite calcular ventajas por token.

El crítico trata de aprender una expectativa por estado, no un número fijo por episodio.
- En el estado justo antes del último token, el crítico aprende a decir: “si sigo desde acá, en promedio espero 0.7 de reward”.
- En un estado más temprano (cuando recién arrancaste a escribir), el crítico podría decir: “desde acá, en promedio espero 0.2 de reward, porque todavía es muy fácil equivocarse”.
- O sea: el crítico no ve solo “un reward concreto”, sino que aprende el promedio esperado desde cada posición a lo largo de muchos episodios distintos. 
- Se entrena en paralelo con su propio value loss.

Como evitar que de mucho advantage a los primeros tokens:
1. Descuento (\gamma<1)
   El crédito que llega desde el final se atenúa hacia atrás: cuanto más lejos esté el token del reward, menos recibe. (Si (\gamma=0.99), a 50 pasos ya cayó mucho).

2. GAE ((\lambda))
   No propagás todo el retorno crudo: usás “sorpresas” (\delta_t) y las filtrás con (\gamma\lambda).
    ⇒ Los primeros tokens reciben poquito y “suavizado”.

3. El crítico aprende “lo esperable” por estado
   Al principio, sí, (V(s)) temprano suele ser bajo. Pero tras ver muchos episodios buenos, el crítico sube su (V(s)) en estados iniciales.
   ⇒ El *advantage* temprano deja de ser grande porque realidad – expectativa se achica.

4. Normalización del advantage
   En práctica se hace A ← (A − mean)/std por minibatch
   ⇒ Evita outliers que dominen el update (ningún tramo de la secuencia “se roba” todo el gradiente).

5. Más “frenos” del policy update

   * Clip PPO corta empujes grandes token-a-token si el ratio (r_t) se va.
   * KL a modelo de referencia otro freno global.
   * Máscaras y pesos podés no dar crédito al *prompt* o bajar el peso a los primeros tokens.



**Ejemplo Importance sampling**

Simple (por trayectoria o outer loop)

In [32]:
import numpy as np
import pandas as pd 

# Distribución objetivo p y de muestreo q
p = {"A": 0.2, "B": 0.8}   # lo que queremos
q = {"A": 0.6, "B": 0.4}   # de donde muestreamos

# f(x) = "gradiente" de cada caso
f = {"A": 10, "B": 0}

# Valor real esperado bajo p
true_expectation = sum(p[x] * f[x] for x in p)
print("Valor real (E_p[f]):", true_expectation)

Valor real (E_p[f]): 2.0


In [33]:
# Muestreamos desde q
N = 1000
samples = np.random.choice(list(q.keys()), size=N, p=list(q.values()))
print(f"Muestras de q:\n{pd.Series(samples).value_counts()}")

Muestras de q:
A    620
B    380
Name: count, dtype: int64


In [34]:
# Promedio ingenuo
naive_estimate = np.mean([f[x] for x in samples])
print("Promedio ingenuo (E_q):", naive_estimate)

Promedio ingenuo (E_q): 6.2


In [35]:
# Importance Sampling (peso w = p/q)
weights = [p[x]/q[x] for x in samples]
# print(f'weights (ratios: p(x)/q(x)): {weights}')
is_estimate = np.mean([w * f[x] for w, x in zip(weights, samples)])
print("Estimador IS (∑ w(x) f(x) / N):", round(is_estimate, 3))

Estimador IS (∑ w(x) f(x) / N): 2.067


Ejemplo mostrando **diferentes tipos de Important sampling**:
- objetivo / ground truth
- IS "bien" por trayectoria
- IS "aprox" por token

In [1]:
import numpy as np
rng = np.random.default_rng(0)  # cambiá la seed si querés

# Largo de secuencia (tokens por trayectoria)
T = 10

ACTIONS = ["A", "B"]  # mantenemos 2 acciones para simpleza

# Políticas por paso (pi_old y pi_new). Cada paso t tiene probs para A/B.
# Podés editarlas a gusto (asegurate que sumen 1 en cada paso).
pi_old = [{"A": 0.8, "B": 0.2} if t % 2 == 0 else {"A": 0.3, "B": 0.7} for t in range(T)]
pi_new = [{"A": 0.5, "B": 0.5} if t % 2 == 0 else {"A": 0.6, "B": 0.4} for t in range(T)]

# Advantages por token (pueden ser constantes o por paso)
A_t = [ +1.0 if t % 2 == 0 else -0.5 for t in range(T) ]  # ej: alterna +1 y -0.5

def ratio_token(t, a):  # r_t = pi_new/pi_old
    return pi_new[t][a] / pi_old[t][a]
pi_old

[{'A': 0.8, 'B': 0.2},
 {'A': 0.3, 'B': 0.7},
 {'A': 0.8, 'B': 0.2},
 {'A': 0.3, 'B': 0.7},
 {'A': 0.8, 'B': 0.2},
 {'A': 0.3, 'B': 0.7},
 {'A': 0.8, 'B': 0.2},
 {'A': 0.3, 'B': 0.7},
 {'A': 0.8, 'B': 0.2},
 {'A': 0.3, 'B': 0.7}]

In [2]:
def seq_prob(policy, seq):
    p = 1.0
    for t, a in enumerate(seq):
        p *= policy[t][a]
    return p

def ground_truth_under_new():
    # E_{tau ~ pi_new}[sum_t A_t]
    # (acá el integrando no depende de la secuencia, así que es la suma A_1+A_2)
    return sum(A_t)

GT = ground_truth_under_new()
print("Ground truth E_{pi_new}[sum A_t] =", GT)

Ground truth E_{pi_new}[sum A_t] = 2.5


In [3]:
N = 16  # cambiá N libremente

def sample_traj_from_old():
    seq = []
    p_old = 1.0
    p_new = 1.0
    for t in range(T):
        a = rng.choice(ACTIONS, p=[pi_old[t]["A"], pi_old[t]["B"]])
        seq.append(a)
        p_old *= pi_old[t][a]
        p_new *= pi_new[t][a]
    return seq, p_old, p_new

batch = [sample_traj_from_old() for _ in range(N)]

print("Primeras 3 trayectorias:")
for i, (seq, po, pn) in enumerate(batch[:3]):
    print(f"{i}: {seq} | p_old={po:.6f} | p_new={pn:.6f}")

Primeras 3 trayectorias:
0: [np.str_('A'), np.str_('A'), np.str_('A'), np.str_('A'), np.str_('B'), np.str_('B'), np.str_('A'), np.str_('B'), np.str_('A'), np.str_('B')] | p_old=0.002529 | p_new=0.000720
1: [np.str_('B'), np.str_('A'), np.str_('B'), np.str_('A'), np.str_('A'), np.str_('A'), np.str_('B'), np.str_('B'), np.str_('A'), np.str_('B')] | p_old=0.000068 | p_new=0.001080
2: [np.str_('A'), np.str_('A'), np.str_('A'), np.str_('B'), np.str_('A'), np.str_('B'), np.str_('B'), np.str_('B'), np.str_('A'), np.str_('B')] | p_old=0.005901 | p_new=0.000480


In [4]:
# Exact IS (traj-level): W_traj = p_new(seq)/p_old(seq)
terms_traj = []
for (seq, p_old, p_new) in batch:
    W_traj = p_new / p_old                 # producto de ratios por token
    inner = sum(A_t)                       # sum_t A_t  (tratamos ∇log como 1 para ver sólo pesos)
    terms_traj.append(W_traj * inner)

est_exact_traj = np.mean(terms_traj)
print("Exact IS (traj-level):", est_exact_traj)


Exact IS (traj-level): 6.23579417360512


In [5]:
terms_token = []
for (seq, p_old, p_new) in batch:
    inner = 0.0
    for t, a in enumerate(seq):
        inner += ratio_token(t, a) * A_t[t]   # r_t * A_t  (∇log ~ 1 para ilustrar)
    terms_token.append(inner)

est_surrogate = np.mean(terms_token)
print("Surrogate (token-level):", est_surrogate)

Surrogate (token-level): 3.247767857142857


In [ ]:
import numpy as np
import pandas as pd
from itertools import product

rng = np.random.default_rng(123)

# === Controls you can edit ===
T_list  = [3, 10, 25, 100]   # sequence lengths to try
N_list  = [3, 10, 25, 100]   # batch sizes to try
reps    = 200                # trials per (T, N)

# Define 2-action policies that alternate structure across time
def make_policies(T):
    # old policy alternates (A-heavy at even t, B-heavy at odd t)
    pi_old = [{"A": 0.8, "B": 0.2} if (t % 2 == 0) else {"A": 0.3, "B": 0.7} for t in range(T)]
    # new policy is more balanced / slightly A-heavy on odd t
    pi_new = [{"A": 0.5, "B": 0.5} if (t % 2 == 0) else {"A": 0.6, "B": 0.4} for t in range(T)]
    return pi_old, pi_new

# Advantages per token (can be any shape you want)
def make_advantages(T):
    # Alternate +1 and -0.5 by time step
    return [1.0 if (t % 2 == 0) else -0.5 for t in range(T)]

ACTIONS = ["A", "B"]

def sample_traj_from_old(pi_old, pi_new):
    seq = []
    p_old = 1.0
    p_new = 1.0
    for t in range(len(pi_old)):
        a = rng.choice(ACTIONS, p=[pi_old[t]["A"], pi_old[t]["B"]])
        seq.append(a)
        p_old *= pi_old[t][a]
        p_new *= pi_new[t][a]
    return seq, p_old, p_new

def ratio_token(pi_old, pi_new, t, a):
    return pi_new[t][a] / pi_old[t][a]

rows = []
for T, N in product(T_list, N_list):
    pi_old, pi_new = make_policies(T)
    A_t = make_advantages(T)

    # Ground truth bajo pi_new: como A_t no depende de la acción, es sum(A_t)
    GT = float(sum(A_t))

    exact_vals = []
    sur_vals   = []
    for _ in range(reps):
        # build one batch
        batch = [sample_traj_from_old(pi_old, pi_new) for _ in range(N)]

        # Exact IS (trajectory-level): weight = product of ratios (p_new/p_old)
        terms_traj = []
        for (seq, p_old, p_new) in batch:
            W_traj = p_new / p_old
            # inner sum over tokens (∇log term ~ 1 solo para comparar pesos)
            inner = sum(A_t)
            terms_traj.append(W_traj * inner)
        exact_vals.append(np.mean(terms_traj))

        # Surrogate (token-level): sum of r_t * A_t; then average over trajectories
        terms_token = []
        for (seq, p_old, p_new) in batch:
            inner = 0.0
            for t, a in enumerate(seq):
                inner += ratio_token(pi_old, pi_new, t, a) * A_t[t]
            terms_token.append(inner)
        sur_vals.append(np.mean(terms_token))

    rows.append({
        "T": T,
        "N": N,
        "GT": GT,
        "Exact_mean": float(np.mean(exact_vals)),
        "Exact_std": float(np.std(exact_vals, ddof=1)),
        "Sur_mean": float(np.mean(sur_vals)),
        "Sur_std": float(np.std(sur_vals, ddof=1)),
        "Exact_bias": float(np.mean(exact_vals) - GT),
        "Sur_bias": float(np.mean(sur_vals) - GT),
    })

df = pd.DataFrame(rows).sort_values(["T","N"]).reset_index(drop=True)
print("Exploración: Exact IS (traj) vs Surrogate (token) según T y N")
df

Exploración: Exact IS (traj) vs Surrogate (token) según T y N
  T   N   GT  Exact_mean  Exact_std  Sur_mean  Sur_std  Exact_bias  Sur_bias
  3   3  1.5    1.515346   1.503303  1.462351 0.602518    0.015346 -0.037649
  3  10  1.5    1.376032   0.624431  1.485491 0.338125   -0.123968 -0.014509
  3  25  1.5    1.478170   0.490886  1.478482 0.212735   -0.021830 -0.021518
  3 100  1.5    1.462910   0.215665  1.496237 0.108755   -0.037090 -0.003763
 10   3  2.5    3.295577  13.788061  2.750149 1.063058    0.795577  0.250149
 10  10  2.5    1.963271   2.152243  2.468437 0.549195   -0.536729 -0.031563
 10  25  2.5    2.245605   2.275499  2.545161 0.359974   -0.254395  0.045161
 10 100  2.5    2.451338   2.042685  2.482991 0.179396   -0.048662 -0.017009
 25   3  7.0    2.679560  10.801364  6.850298 1.713741   -4.320440 -0.149702
 25  10  7.0    5.779369  40.098797  6.920982 0.849592   -1.220631 -0.079018
 25  25  7.0    4.489134  19.168396  6.966018 0.578169   -2.510866 -0.033982
 25 100  7.0  

- Exact trajectory IS = teórico, sin sesgo, pero inviable por varianza (crece mucho con T). Al multiplicar muchos ratios, la **varianza explota** porque hay sub/overflow y el cociente pierde precision.
- Token surrogate = práctico, sesgado, pero con varianza baja y entrenable.

Es el típico trade-off de PPO: preferimos estabilidad aunque perdamos exactitud.

**PPO simplificado** (solo parte de policy clipped. Sin KL ni entropy)

In [56]:
import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)

# Hiperparámetros chiquitos
vocab_size = 5
T = 3
lr = 0.5
eps_clip = 0.2   # zona segura [0.8, 1.2]
advantage = 0.4  # mismo A para todos los pasos (reward final - baseline)

# Modelo lineal sin bias (como el tuyo)
model = nn.Linear(T, vocab_size, bias=False)
with torch.no_grad():
    model.weight.copy_(0.1 * torch.randn(vocab_size, T))

opt = optim.SGD(model.parameters(), lr=lr)
log_softmax = nn.LogSoftmax(dim=-1)

# Estados one-hot (t = 0..T-1)
I = torch.eye(T)

# Elegimos una secuencia de tokens (simula lo que generó π_old)
chosen_tokens = [2, 1, 4]

In [57]:
# ----------------------------------------------------------
# 1) "Foto" de π_old: guardamos logprob_old en esos (s_t, a_t)
# ----------------------------------------------------------
with torch.no_grad():
    logprob_old = []
    for t in range(T):
        logits_t = model(I[t])          # (vocab,)
        logp_t  = log_softmax(logits_t) # (vocab,)
        tok = chosen_tokens[t]
        logprob_old.append(float(logp_t[tok]))
    logprob_old = torch.tensor(logprob_old, dtype=torch.float32)

print("logprob_old por paso:", [f"{x:.4f}" for x in logprob_old.tolist()])

logprob_old por paso: ['-1.7502', '-1.5884', '-1.6374']


In [58]:
# ----------------------------------------------------------
# 2) BEFORE update: logprob_new y ratio (r_t ≈ 1 al inicio)
# ----------------------------------------------------------
with torch.no_grad():
    before = []
    ratio_before = []
    for t in range(T):
        logits_t = model(I[t])
        logp_t  = log_softmax(logits_t)
        tok = chosen_tokens[t]
        logp_new = logp_t[tok]
        r_t = torch.exp(logp_new - logprob_old[t])
        before.append((tok, float(logp_new), float(logp_t.exp()[tok])))
        ratio_before.append(float(r_t))
print("logprob_new BEFORE:", [f"{x[1]:.4f}" for x in before])
print("P_new BEFORE      :", [f"{x[2]:.4f}" for x in before])
print("r_t BEFORE        :", [f"{r:.4f}" for r in ratio_before])

logprob_new BEFORE: ['-1.7502', '-1.5884', '-1.6374']
P_new BEFORE      : ['0.1737', '0.2043', '0.1945']
r_t BEFORE        : ['1.0000', '1.0000', '1.0000']


In [59]:
# ----------------------------------------------------------
# 3) Loss PPO (sólo política, clipeada)
#    L = - mean( min( r_t*A, clamp(r_t)*A ) )
# ----------------------------------------------------------
advantages = torch.full((T,), fill_value=advantage, dtype=torch.float32)

total_unclipped = 0.0
total_clipped = 0.0
for t in range(T):
    logits_t = model(I[t])
    logp_t  = log_softmax(logits_t)
    tok = chosen_tokens[t]
    logp_new = logp_t[tok]
    r_t = torch.exp(logp_new - logprob_old[t])
    unclipped = r_t * advantages[t]
    clipped_r = torch.clamp(r_t, 1.0 - eps_clip, 1.0 + eps_clip)
    clipped = clipped_r * advantages[t]
    total_unclipped = total_unclipped + unclipped
    total_clipped   = total_clipped   + clipped
    print(f"t={t}: tok={tok}  logp_old={logprob_old[t]:+.4f}  logp_new={logp_new:+.4f}  r_t={float(r_t):.4f}  "
          f"uncl={float(unclipped):+.4f}  clp={float(clipped):+.4f}  picked=min(uncl,clp)")


t=0: tok=2  logp_old=-1.7502  logp_new=-1.7502  r_t=1.0000  uncl=+0.4000  clp=+0.4000  picked=min(uncl,clp)
t=1: tok=1  logp_old=-1.5884  logp_new=-1.5884  r_t=1.0000  uncl=+0.4000  clp=+0.4000  picked=min(uncl,clp)
t=2: tok=4  logp_old=-1.6374  logp_new=-1.6374  r_t=1.0000  uncl=+0.4000  clp=+0.4000  picked=min(uncl,clp)


In [60]:
# Promediamos el surrogate (min) y lo negamos para minimizar
surrogate = torch.minimum(total_unclipped, total_clipped) / T
loss = -surrogate
print(f"\nSurrogate (mean): {float(surrogate):+.6f}  ->  PPO policy loss = {float(loss):+.6f}")



Surrogate (mean): +0.400000  ->  PPO policy loss = -0.400000


In [61]:
# Backprop + step
opt.zero_grad()
loss.backward()
opt.step()


In [62]:
# ----------------------------------------------------------
# 4) AFTER update: ver cómo cambiaron P_new de los tokens elegidos
# ----------------------------------------------------------
with torch.no_grad():
    after = []
    ratio_after = []
    for t in range(T):
        logits_t = model(I[t])
        logp_t  = log_softmax(logits_t)
        tok = chosen_tokens[t]
        logp_new = logp_t[tok]
        r_t = torch.exp(logp_new - logprob_old[t])
        after.append((tok, float(logp_new), float(logp_t.exp()[tok])))
        ratio_after.append(float(r_t))

print("\nPer-step chosen token probabilities (before -> after):")
for t, ((tok_b, logpb, pb), (tok_a, logpa, pa)) in enumerate(zip(before, after)):
    assert tok_b == tok_a
    arrow = "↑" if pa > pb else ("↓" if pa < pb else "→")
    print(f"  t={t}: token={tok_b}   P_before={pb:.4f} -> P_after={pa:.4f}   ({arrow})")

print("r_t AFTER         :", [f"{r:.4f}" for r in ratio_after])



Per-step chosen token probabilities (before -> after):
  t=0: token=2   P_before=0.1737 -> P_after=0.1839   (↑)
  t=1: token=1   P_before=0.2043 -> P_after=0.2153   (↑)
  t=2: token=4   P_before=0.1945 -> P_after=0.2052   (↑)
r_t AFTER         : ['1.0583', '1.0540', '1.0552']


## GRPO

Es similar a PPO pero cambia en el **calculo del Advantage**.

Cantidad de trayectorias originales por prompt:
- En **PPO** teniamos varios prompts y una sola respuesta original por prompt. Luego ibamos modificando pi_new prediciendo sobre esa misma trayectoria generada original (una por cada prompt). 
- Ahora en **GRPO** tenemos **k respuestas por prompt**. Tambien vamos modificando pi_new prediciendo sobre las trayectorias generadas originales (k por prompt).

Minibatches:
- En PPO, si en el batch total tenemos 256 prompts, generamos 256 trayectorias y, si el batch_size=16, nos quedaban 16 mini batches
- En GRPO, si en el batch total tenemos 256 prompts, generamos 256*k trayectorias (si k=4 -> 1024 trayectorias). Y si batch_size=16, nos quedan 64 mini batches. O sea, en un minibatch pueden haber dos o mas casos del mismo prompt pero con distintas trayectorias originales.

**Baseline y Advantage**:
- En PPO se calculaba con un critic o value function. Dos formas: TD advantage por paso o GAE (lo clasico). Involucraba otro modelo aprendible lo cual no era bueno.
- En GRPO, tenemos 256 prompts. Para cada prompt generamos k=4 trayectorias -> eso es un GRUPO. Y esas generan rewards. Tomamos la media y desvio de las rewards del grupo y ese es el **baseline**.
    - Lo malo es que ahora no tenemos Advantage por token, solo advantage por trayectoria.

$$A_i = r_i - \bar{r}_{\text{prompt}} \text{ ------ (y opcionalmente} A_i \leftarrow \frac{A_i}{\mathrm{std}}\text{)}$$  

**Objetivo**

$$
J_{\mathrm{GRPO}}(\theta) = \mathbb{E}_{x \sim D,\, \{y_i\}_{i=1}^G \sim \pi_{\mathrm{old}}(\cdot|x)} \left[ 
    \frac{1}{G} \sum_{i=1}^G \frac{1}{|y_i|} \sum_{t=1}^{|y_i|} 
        \min \left( w_{i,t}(\theta) \hat{A}^{i},\; \mathrm{clip}(w_{i,t}(\theta), 1-\epsilon, 1+\epsilon) \hat{A}^{i} \right)
\right]
$$

Simplificando (dejando de lado el clip)

$$
J_{\mathrm{GRPO}}(\theta) = \mathbb{E} \left[ 
    \frac{1}{G} \sum_{i=1}^G \frac{1}{|y_i|} \sum_{t=1}^{|y_i|} 
         w_{i,t}(\theta) \hat{A}^{i}
\right]
$$

Tenemos:
- Dataset de prompts: D = {x}
- Para cada prompt x generamos G completions con pi_old: y_i
- Cada completion y_i es una secuencia de tokens y_i{1:T}

Minibatch:
- Elegimos B prompts del dataset: B
- Para cada x de B, traemos las G completions: y
- Con eso armamos un minibatch: contiene B*G completions, y adentro, tokens

Expectation: En un solo paso de entrenamiento hay 3 niveles para promediar:
- Tokens (inner) dentro de cada completion: promedio o suma de los terminos (ratio*advantage) por token
- Completions del grupo: promedio sobre G
- Prompts del minibatch (outer)

$$
J^{\mathrm{GRPO}}(\theta) = \frac{1}{B} \sum_{b=1}^B \frac{1}{G} \sum_{i=1}^G \frac{1}{T_{b,i}} \sum_{t=1}^{T_{b,i}} 
    \min \left( 
        \frac{\pi_\theta(y_{i,t}^{(b)} \mid x^{(b)}, y_{i,<t}^{(b)})}{\pi_{\mathrm{old}}(y_{i,t}^{(b)} \mid x^{(b)}, y_{i,<t}^{(b)})} \hat{A}^i(b),\;
        \mathrm{clip}(\cdot) \hat{A}^i(b)
    \right)
$$

Detalles implementacion:
- Batching real: se suelen aplanar todos los tokens del batch y se hace un unico promedio por token (con mascara), que es equivalente
- Normalizacion: se puede normalizar por longitud

# Idea: GRPO + TD learning

**¿Qué es?**

Un ejemplo educativo donde un "mini-LLM" genera respuestas **token a token** (sumar `a+b=` en texto) y aprende con **reward solo terminal**: `1` si la respuesta es correcta, `0` si no.

**¿Por qué?**

Los LLMs con RL suelen tener **recompensas escasas y tardías** (solo al final), lo que dificulta:

* **Asignar crédito** a cada token.
* Mantener **estabilidad** (alta varianza en el gradiente).
* Evitar premiar por igual **toda la trayectoria** (buenos y malos pasos).

**¿Qué intentamos resolver?**

Dar **feedback denso por token**, pero **justo por estado** (condicionado al prefijo) y con **menor varianza** que REINFORCE puro o GRPO "plano".

**Idea central**

Usamos una **value head** $$V(s_t)$$ que estima el éxito esperado **desde el estado actual** y construimos un **advantage por token**:

$$
A_t = \underbrace{(r_T - V(s_t))}_{\text{justicia por estado}}
-
\underbrace{\overline{(r_T - V(s_t))}_{\text{grupo}}}_{\text{centrado GRPO coherente}}
+
\beta \underbrace{(V(s_{t+1})-V(s_t))}_{\text{impacto local del token}}
$$

* **Justicia por estado:** solo pagamos la **mejora** sobre lo **esperable** en ese prefijo.
* **Centrado por grupo:** comparamos **K** completions del **mismo prompt** para bajar varianza (todos empiezan del mismo lugar).
* **Impacto local:** cuánto movió el valor ese token concreto.

**¿Cómo se entrena?**

- **Actor (política):** $\mathcal{L}_\pi=-\sum_t A_t\log\pi(a_t|s_t)$ (con *stop-grad* sobre $A_t$).
- **Crítico (valor):** **TD(0)** con reward final: $y_t=V(s_{t+1})$
    - salvo el último paso, donde $y_{T-1}=r_T$. 
    - $\mathcal{L}_V=\sum_t (V(s_t)-y_t)^2$.*(El crítico no se centra por grupo: aprende la expectativa real.)*

**¿Por qué lo enfocamos así?**

* Es **simple**, **interpretables** sus piezas y refleja cómo se usa RL en LLMs:

  * Generación **secuencial**,
  * **Reward terminal** con "checker",
  * **Grupos** por prompt (GRPO),
  * **Baseline** aprendido (value) para reducir varianza y asignar crédito.

**Qué mirar en los logs**

* **Accuracy** media móvil (¿mejora?).
* Por paso: 
    - $V(s_t)$, 
    - $S_t=(r_T-V(s_t))-\overline{(r_T-V(s_t))}$, 
    - $\Delta V_t$, 
    - $A_t$, 
    - y **tokens top-prob**.

**Qué sí logra**

* Señal **densa** y **no plana** por token.
* Menor **varianza** y **mejor crédito** que GRPO puro.
* "**Justicia por estado**": no premia tokens por estar en estados fáciles.

**Qué NO resuelve (limitaciones)**

* La recompensa sigue siendo **tardía** y **ruidosa** (sigue dependiendo de $r_T$).
* Requiere un **buen crítico**; si $V$ es pobre, el advantage se degrada.
* **Horizonte largo** y **exploración** siguen siendo desafíos.

**Próximos pasos naturales**

* **PRMs / checkers** para recompensas intermedias.
* **Return decomposition** (RUDDER-like) para atribuir mérito temprano.
* **Crítico de acción** $Q(s_t,a_t)$ para crédito estado-acción más fino.
* **KL suave** y normalizaciones para mayor estabilidad.



In [ ]:
# RL "LLM-like" toy: adding small numbers as text with sparse terminal reward.
# We show a simple policy-gradient with a value baseline trained by TD(0),
# and a GRPO-style group centering that is "justo por estado":
#   A_t = (r_T - V(s_t)) - mean_group(r_T - V(s_t)) + beta * (V(s_{t+1}) - V(s_t))
#
# The "LLM" is a tiny log-linear policy over hand-crafted features of (prompt, output_prefix, t).
# No backprop-through-time: we use REINFORCE-style score function gradients.
#
# Educational, not optimized. Lots of prints to see what's going on.

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

rng = np.random.default_rng(1)

# -------------------- Task setup --------------------
TOKENS = [str(d) for d in range(10)] + ["<EOS>"]
EOS_IDX = len(TOKENS) - 1
START_IDX = len(TOKENS)  # pseudo-token for "start" in features (not an action)

MAX_OUT = 4  # max output length in tokens (e.g., up to "109<eos>")
MIN_A, MAX_A = 0, 99
MIN_B, MAX_B = 0, 9   # choose small second addend to keep problem easy-ish

def prompt_to_str(a, b):
    return f"{a}+{b}="

def eval_terminal_reward(pred_tokens, a, b):
    # pred_tokens: list of action indices (0..10), stop at EOS or MAX_OUT
    out_digits = []
    for idx in pred_tokens:
        if idx == EOS_IDX:
            break
        out_digits.append(TOKENS[idx])
    out_str = "".join(out_digits)
    correct = str(a + b)
    return 1.0 if out_str == correct else 0.0, out_str, correct

In [ ]:
# -------------------- Featurization --------------------
# Features x(s_t) = [bias, a_norm, b_norm, onehot(t in 0..MAX_OUT-1), onehot(prev_token in 0..EOS+START)]
FEAT_DIM = 1 + 2 + MAX_OUT + (len(TOKENS) + 1)  # bias + a,b + tOH + prevOH

def featurize(a, b, t, prev_token_idx):
    x = np.zeros(FEAT_DIM, dtype=np.float64)
    off = 0
    # bias
    x[off] = 1.0; off += 1
    # normalized a,b
    x[off] = a / MAX_A; off += 1
    x[off] = b / MAX_B if MAX_B > 0 else 0.0; off += 1
    # t one-hot
    if 0 <= t < MAX_OUT:
        x[off + t] = 1.0
    off += MAX_OUT
    # prev token one-hot over size (len(TOKENS) + 1) including START
    if prev_token_idx is None:
        idx = START_IDX
    else:
        idx = prev_token_idx
    x[off + idx] = 1.0
    # done
    return x

# -------------------- Models (policy & value) --------------------
@dataclass
class Policy:
    W: np.ndarray  # shape [FEAT_DIM, n_actions]

    @staticmethod
    def init():
        # small random init for exploration
        return Policy(W=0.01 * rng.standard_normal((FEAT_DIM, len(TOKENS))))

    def logits(self, x):
        return x @ self.W  # [n_actions]

    def probs(self, x):
        z = self.logits(x)
        z = z - np.max(z)  # stabilize
        p = np.exp(z)
        p /= p.sum()
        return p

    def sample(self, x):
        p = self.probs(x)
        a = rng.choice(len(TOKENS), p=p)
        return a, p

    def grad_logpi(self, x, a_idx, p):
        # score function gradient wrt W: x ⊗ (onehot(a) - p)
        onehot = np.zeros_like(p)
        onehot[a_idx] = 1.0
        return np.outer(x, onehot - p)


@dataclass
class ValueFn:
    w: np.ndarray  # shape [FEAT_DIM]

    @staticmethod
    def init():
        return ValueFn(w=np.zeros(FEAT_DIM, dtype=np.float64))

    def V(self, x):
        return float(x @ self.w)

    def update(self, x, target, lr):
        v = self.V(x)
        self.w += -2.0 * lr * (v - target) * x  # grad of (v - target)^2


# -------------------- Rollout --------------------
@dataclass
class Step:
    x: np.ndarray      # features at s_t
    a_idx: int         # action taken
    p: np.ndarray      # probs at s_t
    t: int             # position in output
    prev_token: int    # previous token index used in features

@dataclass
class Trajectory:
    steps: list
    r_T: float
    out_str: str
    correct_str: str

def rollout(policy, a, b):
    steps = []
    prev_token = None  # START
    out_tokens = []
    for t in range(MAX_OUT):
        x = featurize(a, b, t, prev_token)
        a_idx, p = policy.sample(x)
        steps.append(Step(x=x, a_idx=a_idx, p=p, t=t, prev_token=START_IDX if prev_token is None else prev_token))
        out_tokens.append(a_idx)
        if a_idx == EOS_IDX:
            break
        prev_token = a_idx
    r_T, out_str, correct = eval_terminal_reward(out_tokens, a, b)
    return Trajectory(steps=steps, r_T=r_T, out_str=out_str, correct_str=correct)

# -------------------- Training Loop --------------------
def train(
    epochs=1200,
    K=4,
    lr_actor=0.05,
    lr_value=0.08,
    beta=0.25,
    td_kind="TD",  # "TD" or "MC"
    print_every=50,
):
    pol = Policy.init()
    val = ValueFn.init()

    acc_hist = []
    debug_every = max(100, print_every)

    for ep in range(1, epochs+1):
        # Sample a random prompt
        a = int(rng.integers(MIN_A, MAX_A+1))
        b = int(rng.integers(MIN_B, MAX_B+1))

        # K completions (group)
        group = [rollout(pol, a, b) for _ in range(K)]

        # Accuracy logging
        acc_hist.extend([traj.r_T for traj in group])

        # Compute rv_t = (r_T - V(s_t)) for each step in each traj
        # We need group mean per time index t (only over members that reached t)
        max_T = max(len(traj.steps) for traj in group)
        rv_by_t = [[] for _ in range(max_T)]
        for traj in group:
            for t, step in enumerate(traj.steps):
                rv_by_t[t].append(traj.r_T - val.V(step.x))
        rv_mean_by_t = [np.mean(lst) if lst else 0.0 for lst in rv_by_t]

        # --- Policy update ---
        for traj in group:
            Tlen = len(traj.steps)
            for t, step in enumerate(traj.steps):
                # Global, justo por estado, centrado por grupo en el mismo t
                S_t = (traj.r_T - val.V(step.x)) - rv_mean_by_t[t]
                # Impacto local
                if t < Tlen - 1:
                    dV_t = val.V(traj.steps[t+1].x) - val.V(step.x)
                else:
                    dV_t = 0.0
                A_t = S_t + beta * dV_t

                # Policy gradient
                grad_W = pol.grad_logpi(step.x, step.a_idx, step.p)
                pol.W += lr_actor * A_t * grad_W

        # --- Value update (critic) ---
        for traj in group:
            Tlen = len(traj.steps)
            for t, step in enumerate(traj.steps):
                if td_kind == "TD":
                    target = val.V(traj.steps[t+1].x) if t < Tlen - 1 else traj.r_T
                else:  # Monte Carlo target
                    target = traj.r_T
                val.update(step.x, target, lr_value)

        # Prints / debug
        if ep % print_every == 0:
            window = min(400, len(acc_hist))
            avg = np.mean(acc_hist[-window:]) if window > 0 else 0.0
            print(f"[ep {ep:4d}] prompt '{prompt_to_str(a,b)}'  recent-avg-acc over {window} rollouts: {avg:.3f}")

        if ep % debug_every == 0:
            # Pick the first traj for detailed print
            dt = group[0]
            print("\n=== DEBUG SNAPSHOT ===")
            print(f"Prompt: '{prompt_to_str(a,b)}'")
            for i, tr in enumerate(group):
                print(f"  member {i}: out='{tr.out_str}'  correct='{tr.correct_str}'  r_T={tr.r_T:.1f}")
            print("  Per-step details for member 0:")
            for t, step in enumerate(dt.steps):
                Vst = val.V(step.x)
                S_t = (dt.r_T - Vst) - rv_mean_by_t[t]
                if t < len(dt.steps)-1:
                    dV_t = val.V(dt.steps[t+1].x) - Vst
                else:
                    dV_t = 0.0
                A_t = S_t + beta * dV_t
                p = step.p
                top2 = np.argsort(-p)[:2]
                print(f"   t={t} prevTok={'START' if step.prev_token==START_IDX else TOKENS[step.prev_token]}  "
                      f"act='{TOKENS[step.a_idx]}'  V(s_t)={Vst:.3f}  S_t={S_t:.3f}  dV_t={dV_t:.3f}  A_t={A_t:.3f}  "
                      f"top={[(TOKENS[j], float(p[j])) for j in top2]}")
            print("======================\n")

    return pol, val, acc_hist



In [ ]:
# Plot moving accuracy
def moving_avg(x, n=200):
    x = np.array(x, dtype=float)
    if len(x) < n:
        return x
    c = np.cumsum(x)
    c[n:] = c[n:] - c[:-n]
    return c[n-1:] / n

ma = moving_avg(acc_hist, n=200)
plt.figure()
plt.plot(ma)
plt.title("Moving accuracy (window=200 rollouts)")
plt.xlabel("Rollout index")
plt.ylabel("Accuracy")
plt.show()


In [ ]:
pol, val, acc_hist = train(
    epochs=1500,   # keep modest so it runs quickly
    K=4,
    lr_actor=0.05,
    lr_value=0.05,
    beta=0.3,
    td_kind="TD",
    print_every=75,
)

In [ ]:
# Quick eval on a few fixed prompts
def eval_prompt(a, b, samples=5):
    print(f"\nEVAL '{prompt_to_str(a,b)}'")
    for i in range(samples):
        # greedy for display: take argmax to see the deterministic best guess
        prev = None
        out_tokens = []
        for t in range(MAX_OUT):
            x = featurize(a,b,t,prev)
            p = pol.probs(x)
            a_idx = int(np.argmax(p))
            out_tokens.append(a_idx)
            if a_idx == EOS_IDX:
                break
            prev = a_idx
        r_T, out_str, correct = eval_terminal_reward(out_tokens, a, b)
        print(f"  sample {i}: out='{out_str}'  correct='{correct}'  r_T={r_T}  first-step-top3=",
              sorted([(TOKENS[j], float(pj)) for j,pj in enumerate(p)], key=lambda z:-z[1])[:3])

for (a,b) in [(3,5),(12,7),(37,5),(99,9),(48,0)]:
    eval_prompt(a,b, samples=3)
